In [2]:
import difflib, os, re, requests, yaml

import pandas as pd
import ollama

from box import Box
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from openai import OpenAI


def normalize_text(text):
    return re.sub(r'\s+', ' ', text).lower().strip()

def find_similar_specialties(specialty_list, text, cutoff=0.75):
    normalized_text = normalize_text(text)
    normalized_specialties = [normalize_text(s) for s in specialty_list]

    # First try exact match
    for original, normalized in zip(specialty_list, normalized_specialties):
        if normalized == normalized_text:
            return [original]

    # Then try fuzzy matching
    close_matches = difflib.get_close_matches(normalized_text, normalized_specialties, n=1, cutoff=cutoff)
    if close_matches:
        matched_index = normalized_specialties.index(close_matches[0])
        return [specialty_list[matched_index]]

    return None


def extract_specialty_from_response(response_text):
    # Try strict extraction first
    match = re.search(r'Specialty:\s*(.+)', response_text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    # Fallback: try to guess from any line that looks like a specialty
    lines = response_text.strip().splitlines()
    for line in lines:
        line = line.strip()
        if line and not line.lower().startswith("patient") and not line.lower().startswith("task"):
            return line
    return response_text.strip()


specialty_list = [
        # 'Allergy / Immunology',
        'Cardiovascular / Pulmonary',
        'Orthopedic',
        # 'Radiology',
        'Urology',
        'ENT - Otolaryngology',
        'Ophthalmology',
        'Psychiatry / Psychology',
        'Dermatology',
        # 'General Medicine',
        'Sleep Medicine',
        # 'Rheumatology',
        # 'Nephrology',
        # 'Hematology - Oncology',
        'Gastroenterology',
        # 'Endocrinology',
        'Obstetrics / Gynecology', 'Neurology / Neurosurgery', 'Podiatry'
    ]

response_text = "Specialty: Neurology / Neurosurgery"
spec_str = extract_specialty_from_response(response_text)

ans = find_similar_specialties(specialty_list, spec_str, cutoff=0.8)
predicted_specialty = "none" if ans is None or len(ans) > 1 else ans[0]

predicted_specialty, response_text



('Neurology / Neurosurgery', 'Specialty: Neurology / Neurosurgery')

In [3]:
import difflib, os, re, requests, yaml

import pandas as pd
import ollama

from box import Box
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from openai import OpenAI


def find_similar_specialties(specialty_list, text, cutoff=0.6):
    text_lower = text.lower()
    found = []
    
    for specialty in specialty_list:
        specialty_lower = specialty.lower()
        
        # Exact match first
        if specialty_lower in text_lower:
            found.append(specialty)
            continue

        # Approximate match: compare specialty against text chunks (sliding window)
        text_words = text_lower.split()
        spec_words = specialty_lower.split()
        window_size = len(spec_words)

        # Sliding window over the text
        for i in range(len(text_words) - window_size + 1):
            text_chunk = ' '.join(text_words[i:i+window_size])
            similarity = difflib.SequenceMatcher(None, specialty_lower, text_chunk).ratio()
            if similarity >= cutoff:
                found.append(specialty)
                break

    if not found:
        return None
    return list(set(found))


def extract_specialty_from_response(response_text):
    # Try strict extraction first
    match = re.search(r'Specialty:\s*(.+)', response_text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    # Fallback: try to guess from any line that looks like a specialty
    lines = response_text.strip().splitlines()
    for line in lines:
        line = line.strip()
        if line and not line.lower().startswith("patient") and not line.lower().startswith("task"):
            return line
    return response_text.strip()


specialty_list = [
        # 'Allergy / Immunology',
        'Cardiovascular / Pulmonary',
        'Orthopedic',
        # 'Radiology',
        'Urology',
        'ENT - Otolaryngology',
        'Ophthalmology',
        'Psychiatry / Psychology',
        'Dermatology',
        # 'General Medicine',
        'Sleep Medicine',
        # 'Rheumatology',
        # 'Nephrology',
        # 'Hematology - Oncology',
        'Gastroenterology',
        # 'Endocrinology',
        'Obstetrics / Gynecology', 'Neurology / Neurosurgery', 'Podiatry'
    ]

response_text = "Specialty: Neurology / Neurosurgery"
spec_match = re.search(r'Specialty:\s*([^\n\r]+)', response_text, re.IGNORECASE)
spec_str = spec_match.group(1).strip() if spec_match else response_text.strip()



ans = find_similar_specialties(specialty_list, spec_str, cutoff=0.8)
predicted_specialty = "none" if ans is None or len(ans) > 1 else ans[0]

predicted_specialty, response_text



('none', 'Specialty: Neurology / Neurosurgery')

In [16]:
import os
import glob
import html
import markdown

root_folder = "res/ai-discharge/2025_07_07_07_24_32_44_discharge_gpt-4.1/medical-cases"
output_html = "res/ai-discharge/2025_07_07_07_24_32_44_discharge_gpt-4.1/all_cases_gpt-4.1.html"
basename    = "gpt-4.1"


def read_file(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except Exception:
        return None

sections = []
case_folders = sorted([f for f in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, f))])

for case_id in case_folders:
    case_path = os.path.join(root_folder, case_id)
    report_md = os.path.join(case_path, f"{case_id}_user_friendly_report_en.md")
    record_en = os.path.join(case_path, f"{case_id}_orig_medical_record_en.txt")
    record_zh = os.path.join(case_path, f"{case_id}_orig_medical_record_zh.txt")
    
    report_content_raw = read_file(report_md)
    if report_content_raw is not None:
        report_content = markdown.markdown(report_content_raw)
    else:
        report_content = "<i>Missing or unreadable markdown</i>"

    record_en_content_raw = read_file(record_en)
    record_en_content = (
        f"<pre>{html.escape(record_en_content_raw)}</pre>" if record_en_content_raw is not None else "<i>Missing or unreadable EN text</i>"
    )
    record_zh_content_raw = read_file(record_zh)
    record_zh_content = (
        f"<pre>{html.escape(record_zh_content_raw)}</pre>" if record_zh_content_raw is not None else "<i>Missing or unreadable ZH text</i>"
    )

    section_html = f"""
    <section style="margin-bottom: 40px;">
      <h2>{basename} - Case ID: {case_id}</h2>
      <div class="case-content case-flex">
        <div class="case-col">
          <h3>User Friendly Report (Markdown)</h3>
          {report_content}
        </div>
        <div class="case-col">
          <h3>Original Medical Record (EN)</h3>
          {record_en_content}
        </div>
        <div class="case-col">
          <h3>Original Medical Record (ZH)</h3>
          {record_zh_content}
        </div>
      </div>
    </section>
    """
    sections.append(section_html)

full_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>All Cases: Side by Side Comparison</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background: inherit;
            margin: 0;
            padding: 0 2vw;
        }}
        h1 {{
            color: #245;
            margin-bottom: 2rem;
        }}
        h2 {{
            margin-top: 32px;
            margin-bottom: 8px;
            color: #245;
            border-bottom: 1px solid #ccd;
            padding-bottom: 4px;
        }}
        h3 {{
            margin-top: 0;
            font-size: 1.1em;
            color: #444;
            border-bottom: 1px solid #eee;
        }}
        section {{
            background: #fff;
            border-radius: 7px;
            box-shadow: 0 2px 6px #0001;
            padding: 12px 8px 16px 8px;
        }}
        .case-flex {{
            display: flex;
            gap: 16px;
        }}
        .case-col {{
            flex: 1;
            border: 1px solid #ccc;
            padding: 8px;
            min-width: 0;
            background: #fff;
            overflow-x: auto;
            max-width: 33vw;
            max-height: 600px;
        }}
        /* --- High contrast fix only for case content --- */
        .case-content, .case-content * {{
            color: #222 !important;
            background: #fff !important;
        }}
        /* For markdown formatting in the report */
        .case-content ul, .case-content ol {{
            margin: 0 0 0 20px;
            padding: 0 0 0 18px;
        }}
        .case-content li {{
            margin-bottom: 6px;
        }}
        .case-content strong, .case-content b {{
            font-weight: bold !important;
            color: #111 !important;
        }}
        .case-content em, .case-content i {{
            font-style: italic !important;
        }}
        pre {{
            white-space: pre-wrap;
            word-break: break-all;
            font-family: monospace;
            font-size: 14px;
            margin: 0;
        }}
        @media (max-width: 900px) {{
            .case-flex {{
                flex-direction: column;
            }}
            .case-col {{
                max-width: 100vw;
            }}
        }}
    </style>
</head>
<body>
    <h1>All Cases: Side by Side Comparison</h1>
    {''.join(sections)}
</body>
</html>
"""

with open(output_html, "w", encoding="utf-8") as f:
    f.write(full_html)

print(f"Done. Output written to {output_html}")

Done. Output written to res/ai-discharge/2025_07_07_07_24_32_44_discharge_gpt-4.1/all_cases_gpt-4.1.html


In [8]:
import os

root_folder = "res/ai-discharge/2025_07_07_07_24_32_44_discharge_gpt-4.1/medical-cases"

for case_id in os.listdir(root_folder):
    subfolder = os.path.join(root_folder, case_id)
    if os.path.isdir(subfolder):
        for filename in os.listdir(subfolder):
            # Skip if file already starts with case_id_
            if filename.startswith(f"{case_id}_"):
                continue
            old_path = os.path.join(subfolder, filename)
            new_filename = f"{case_id}_{filename}"
            new_path = os.path.join(subfolder, new_filename)
            os.rename(old_path, new_path)
            # print(f"Renamed: {old_path} --> {new_path}")


In [3]:
import json

# Path to your file
file_path = 'datasets/patients.json'

with open(file_path, 'r') as f:
    cases = json.load(f)
print(len(cases))

506


In [ ]:

def get_nested_keys(d, prefix=''):
    keys = set()
    for k, v in d.items():
        full_key = f"{prefix}.{k}" if prefix else k
        keys.add(full_key)
        if isinstance(v, dict):
            keys |= get_nested_keys(v, prefix=full_key)
    return keys

reference_nested = get_nested_keys(cases[0])
print(reference_nested)

for idx, case in enumerate(cases):
    nested = get_nested_keys(case)
    if nested != reference_nested:
        print(f"Case {idx} nested keys differ!")
        print(nested)
        break


{'raw_medical_record.鉴别诊断', 'comment_num', 'medical_record.现病史', 'id', 'title', 'medical_record.初步诊断', 'raw_medical_record.诊断结果', 'medical_record.诊治经过', 'medical_record.鉴别诊断', 'medical_record.主诉', 'local', 'medical_record.查体', 'medical_record.诊断结果', 'profile', 'raw_medical_record.现病史', 'raw_medical_record.分析总结', 'raw_medical_record', 'author', 'reformed_text_medical_record', 'raw_medical_record.诊断依据', 'url', 'medical_record.分析总结', 'medical_record.一般资料', 'medical_record.辅助检查', 'medical_record.诊断依据', 'raw_medical_record.辅助检查', 'raw_medical_record.既往史', 'raw_medical_record.诊治经过', 'department', 'diseases', 'disease_info', 'raw_medical_record.初步诊断', 'raw_medical_record.一般资料', 'raw_medical_record.主诉', 'medical_record.既往史', 'read_num', 'time', 'raw_medical_record.查体', 'medical_record'}
Case 2 nested keys differ!
{'raw_medical_record.鉴别诊断', 'comment_num', 'medical_record.现病史', 'id', 'title', 'medical_record.初步诊断', 'raw_medical_record.诊断结果', 'medical_record.诊治经过', 'medical_record.鉴别诊断', 'medica